# Evaluation Methodology

> model, , , score, "". evaluation"model", score.
>
> evaluation: evaluationmetric, open-sourceevaluationframework, LLM-as-Judge a strong modeljudge, resultcomparereport.

 (benchmark) , (metric) pipeline (evaluation pipeline) . LLM-as-Judge : GPT-4 a strong modelmodel. MT-Bench, Chatbot Arena , a strong model judge consistency; consistency, judge prompt, modelversionanswer. : [MT-Bench / Chatbot Arena](https://arxiv.org/abs/2306.05685).

LLM-as-Judge ——, .

Perplexity model token . pre-training, , inference. 

In [ ]:
import sys

print(f"Python: {sys.version.split()[0]}")

# check
deps = ["openai", "datasets", "lm_eval"]
for pkg in deps:
 try:
 __import__(pkg)
 print(f" {pkg}: ")
 except ImportError:
 print(f" {pkg}: (pip install {pkg})")

## 1. The Evaluation Landscape

### 1.1 evaluation ()

```
2019-2021 2022-2023 2024-2025
 ↓ ↓ ↓
GLUE/SuperGLUE MMLU/GSM8K LLM-as-Judge
BERT GPT-4 Agent
 + generation + +
```

### 1.2 evaluation

| | dataset | metric | |
|:---|:---|:---|:---|
| **** | MMLU-Pro, GPQA | acc | modelreport |
| **inference** | GSM8K, MATH, AIME 2024 | exact_match | inferencemetric |
| **** | HumanEval+, LiveCodeBench, SWE-bench | pass@k | |
| **** | IFEval, MT-Bench | strict_acc | |
| **** | AlpacaEval, Chatbot Arena | win_rate/Elo | a strong modeljudge |
| **** | TruthfulQA, Garak | | |
| **** | Needle-in-Haystack, RULER | recall | RAG |
| **** | CMMLU, C-Eval | acc | |
| **Agent** | SWE-bench, WebArena | success_rate | |

### 1.3 modelevaluation

modelreport, model. model, RAG , /model:

```
: MMLU-Pro + GPQA + HellaSwag
: HumanEval+ + LiveCodeBench + SWE-bench (Agent )
: GSM8K + MATH + AIME 2024
: AlpacaEval 2.0 / Chatbot Arena Elo
: IFEval + MT-Bench
: TruthfulQA +
```

**evaluation** ( 4 ) : GSM8K → MMLU → HumanEval → IFEval

## 2. Core Evaluation Frameworks & Recommended Repos

evaluationframework. , :

### 2.1 lm-evaluation-harness (EleutherAI) — open-sourceevaluationframework

```bash
git clone https://github.com/EleutherAI/lm-evaluation-harness.git
cd lm-evaluation-harness
pip install -e .
```

- ****: open-sourcemodelevaluationframework, leaderboard /metric. compare prompt, few-shot, modelversion. : [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
- ****: 200+ dataset, API evaluation / HF evaluation / vLLM evaluation
- **OpenAI-Compatible **: `local-completions` `local-chat-completions` model, API

### 2.2 AlpacaEval — LLM-as-Judge

```bash
git clone https://github.com/tatsu-lab/alpaca_eval.git
cd alpaca_eval
pip install -e .
```

- ****: evaluation, length-controlled win rate; evaluation. : [AlpacaEval](https://github.com/tatsu-lab/alpaca_eval), [LC AlpacaEval](https://arxiv.org/abs/2404.04475)
- **metric**: LC Win Rate (Length-Controlled, bias) , WR ()
- **800+ prompt**, comparemodel GPT-4/Davinci-003 , GPT-4

### 2.3 FastChat (LMSYS) — Chatbot Arena

```bash
git clone https://github.com/lm-sys/FastChat.git
cd FastChat
pip install -e ".[eval]"
```

- ****: Chatbot Arena evaluation; FastChat MT-Bench judge
- ****: MT-Bench (80 + GPT-4 ) , Chatbot Arena (1M+ )

### 2.4 DeepEval — CI/CD

```bash
pip install deepeval
```

- ****: pytest , CI/CD
- ****: , answer, , G-Eval (evaluation)

### framework

| | recommendedframework |
|:---|:---|
| ** / open-sourcemodelevaluation** | lm-evaluation-harness |
| **evaluation** | AlpacaEval / FastChat MT-Bench |
| **CI/CD evaluation** | DeepEval / Promptfoo |
| **** | Garak (NVIDIA) |
| **Agent evaluation** | SWE-bench + WebArena |

** Part **: lm-evaluation-harness evaluation, AlpacaEval evaluation. 

## 3. Hands-On Evaluation via the OpenAI-Compatible API

evaluation——deployment OpenAI API serving (vLLM, Ollama, DeepSeek, ) , API evaluation.

### 3.1

lm-evaluation-harness model OpenAI-Compatible API:

| model | API | |
|:---|:---|:---|
| `local-chat-completions` | `/v1/chat/completions` | generation (GSM8K, HumanEval, IFEval) |
| `local-completions` | `/v1/completions` | generation + (MMLU logprobs) |

****: MMLU / HellaSwag token logprob. Chat Completions API logprobs, completions/logits ; generation, score leaderboard .

### 3.2 serving

 OpenAI API serving:

```
OpenAI API → API Key
DeepSeek API → OpenAI
vLLM deployment → http://localhost:8000/v1
Ollama → http://localhost:11434/v1
LiteLLM →
SGLang → http://localhost:30000/v1
```

****: serving `POST /v1/chat/completions`, generation; /loglikelihood completions/logprobs logits. OpenAI-compatible serving logprobs, chat template, stop, reasoning_content . 

GSM8K generation, Chat Completions API evaluation. , `base_url`, `model` `token` :

```bash
# API (OpenAI)
lm_eval --model local-chat-completions \
 --model_args model=gpt-4o-mini,base_url=https://api.openai.com/v1/chat/completions,token=$OPENAI_API_KEY,num_concurrent=4,max_retries=3,tokenized_requests=False \
 --tasks gsm8k --batch_size 8 \
 --output_path ./eval_results/gsm8k_openai

# API (DeepSeek, OpenAI )
lm_eval --model local-chat-completions \
 --model_args model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4,max_retries=3,tokenized_requests=False \
 --tasks gsm8k --batch_size 8 \
 --output_path ./eval_results/gsm8k_deepseek

# deployment (vLLM)
lm_eval --model local-chat-completions \
 --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/chat/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
 --tasks gsm8k --batch_size 8 \
 --output_path ./eval_results/gsm8k_vllm

# deployment (Ollama)
lm_eval --model local-chat-completions \
 --model_args model=llama3,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
 --tasks gsm8k --batch_size 8 \
 --output_path ./eval_results/gsm8k_ollama
```

 `pip install lm-eval` . 

MMLU , loglikelihood evaluation () , Completions API logprobs. Chat Completions API loglikelihood `NotImplementedError`.

```bash
# vLLM /v1/completions logprobs, MMLU
lm_eval --model local-completions \
 --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/completions,num_concurrent=4,max_retries=3,tokenized_requests=False \
 --tasks mmlu --batch_size 16 \
 --output_path ./eval_results/mmlu
```

Note:

- OpenAI `gpt-4o-mini` `/v1/completions`, `davinci-002` model
- DeepSeek `/v1/completions` , generation Chat API
- Chat API, MMLU generationversion (modeloutput A/B/C/D) , score leaderboard compare

hands-on: vLLM deploymentopen-sourcemodel, MMLU `local-completions`, GSM8K `local-chat-completions`, . 

### 3.3 Batch-Evaluating Multiple Datasets (Python API)

CLI , notebook datasetevaluation, Python API . benchmark, dataset few-shot , result DataFrame .

lm-eval Python API `simple_evaluate` , CLI , Python , visualize. batch benchmark. 

In [ ]:
# lm_eval Python API batchevaluation
# CLI , Python API Notebook evaluation

import os

try:
 from lm_eval import simple_evaluate
 HAS_LMEVAL = True
except ImportError:
 HAS_LMEVAL = False
 print("lm_eval , simple_evaluate result")
 print("run: pip install lm-eval\n")

HAS_DEEPSEEK_KEY = bool(os.environ.get("DEEPSEEK_API_KEY"))

if HAS_LMEVAL and HAS_DEEPSEEK_KEY:
 # run ( API Key model)
 results = simple_evaluate(
 model="local-chat-completions",
 model_args="model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4",
 tasks=["gsm8k", "ifeval"],
 batch_size=8,
 )
 for task, metrics in results["results"].items():
 print(f"{task}: {metrics}")
else:
 if HAS_LMEVAL and not HAS_DEEPSEEK_KEY:
 print("lm_eval , DEEPSEEK_API_KEY; result, API. ")
 # output (report, )
 example_results = {
 "gsm8k": {"exact_match,strict-match": 0.834, "exact_match,flexible-extract": 0.871},
 "mmlu": {"acc,none": 0.743, "acc_norm,none": 0.725},
 "hellaswag": {"acc,none": 0.829, "acc_norm,none": 0.841},
 "ifeval": {"prompt_level_strict_acc,none": 0.687, "inst_level_strict_acc,none": 0.763},
 "humaneval": {"pass@1": 0.689},
 }
 for task, metrics in example_results.items():
 print(f"{task}:")
 for metric, value in metrics.items():
 print(f" {metric}: {value:.4f}")

## 4. LLM-as-Judge: A Strong Model as the Judge

answer, ""—— **LLM-as-Judge** .

### 4.1

```
model ──→ GPT-4 ──→ compare baseline ──→ (Win Rate)
baseline ──→ ──→ ──→
```

framework:
- **AlpacaEval 2.0**: 805 prompt, model vs GPT-4, GPT-4 Turbo judge
- **MT-Bench**: 80 , GPT-4 (1-10 )

### 4.2

| metric | | |
|:---|:---|:---|
| **Win Rate (WR)** | baseline | GPT-4 baseline ≈ 50% |
| **LC Win Rate** | Length-Controlled WR, bias | WR , "" |
| **MT-Bench Score** | GPT-4 1-10 | 7+ , 8+ |
| **Elo Rating** | Elo , | Chatbot Arena output |

### 4.3 hands-on: OpenAI SDK LLM-as-Judge

 judge — framework. 

In [ ]:
# OpenAI SDK LLM-as-Judge
# : , model, prompt, GPT-4

JUDGE_PROMPT = """Please act as an impartial judge and evaluate the quality of the response provided by an AI assistant to the user question displayed below.

Your evaluation should consider the following factors:
1. Helpfulness: Does the response address the user's needs?
2. Accuracy: Is the information factually correct?
3. Relevance: Does the response stay on topic?
4. Depth: Does it provide meaningful detail?
5. Creativity: Is the response well-structured and clear?

Begin your evaluation by providing a short explanation. Be as objective as possible.
After providing your explanation, you must rate the response on a scale of 1 to 10 by strictly following this format:
"[[rating]]", for example: "Rating: [[7]]".

[Question]
{question}

[The Start of Assistant's Answer]
{answer}
[The End of Assistant's Answer]"""

# evaluationsample (MT-Bench )
eval_samples = [
 {
 "question": "Write a Python function to find the longest common subsequence of two strings.",
 "good_answer": "Here's a Python implementation of LCS using dynamic programming:\n\n```python\ndef lcs(s1: str, s2: str) -> str:\n m, n = len(s1), len(s2)\n dp = [[\"\"] * (n + 1) for _ in range(m + 1)]\n for i in range(1, m + 1):\n for j in range(1, n + 1):\n if s1[i-1] == s2[j-1]:\n dp[i][j] = dp[i-1][j-1] + s1[i-1]\n else:\n dp[i][j] = max(dp[i-1][j], dp[i][j-1], key=len)\n return dp[m][n]\n```\n\nTime complexity: O(mn), Space: O(mn).",
 "bad_answer": "def lcs(s1, s2):\n return ''.join(c for c in s1 if c in s2)",
 },
 {
 "question": "Explain the concept of quantum entanglement in simple terms.",
 "good_answer": "Quantum entanglement is when two particles become linked in such a way that the state of one instantly influences the state of the other, no matter how far apart they are. Imagine two magic coins: when you flip them, if one shows heads, the other always shows tails — even if they're on opposite sides of the universe. Einstein called this 'spooky action at a distance' because it seems to violate the idea that nothing can travel faster than light. Today, entanglement is a proven phenomenon and forms the basis for quantum computing and quantum cryptography.",
 "bad_answer": "Quantum entanglement is when two things are connected. It's like twins who can feel each other's pain. Scientists use it for computers.",
 },
]

def judge_with_gpt(question, answer):
 """ GPT-4 ( OPENAI_API_KEY ) """
 try:
 from openai import OpenAI
 client = OpenAI() # OPENAI_API_KEY
 response = client.chat.completions.create(
 model="gpt-4o-mini",
 messages=[{
 "role": "user",
 "content": JUDGE_PROMPT.format(question=question, answer=answer)
 }],
 temperature=0,
 max_tokens=512,
 )
 return response.choices[0].message.content
 except Exception as e:
 return f"[Judge unavailable: {e}]"

import re
for i, sample in enumerate(eval_samples):
 print(f" {i+1}: {sample['question']}")
 print(f"{'-'*60}")
 for label, answer in [("Good Answer", sample['good_answer']), ("Bad Answer", sample['bad_answer'])]:
 verdict = judge_with_gpt(sample['question'], answer)
 match = re.search(r'\[\[(\d+(?:\.\d+)?)\]\]', verdict)
 if match:
 print(f" {label}: {match.group(1)}/10")
 else:
 preview = verdict[:150].replace('\n', ' ')
 print(f" {label}: {preview}...")
 print()

print(": N prompt × M model × GPT-4 judge = LLM-as-Judge evaluation")
print(" MT-Bench AlpacaEval . ")

## 5. Aggregating and Comparing Evaluation Results

In [ ]:
# evaluationresultaggregatevisualize
# : aggregatevisualize; report, modelversionevaluation
benchmark_results = {
 "GPT-4o": {
 "MMLU": 88.7, "GSM8K": 96.1, "HumanEval": 90.2, "HellaSwag": 95.3,
 "IFEval": 84.3, "GPQA": 53.6, "AlpacaEval LC": 57.5,
 },
 "DeepSeek-V3 (671B)": {
 "MMLU": 88.5, "GSM8K": 95.3, "HumanEval": 82.6, "HellaSwag": 89.0,
 "IFEval": 86.1, "GPQA": 59.1, "AlpacaEval LC": 54.2,
 },
 "Qwen2.5-72B": {
 "MMLU": 86.1, "GSM8K": 91.6, "HumanEval": 86.6, "HellaSwag": 86.9,
 "IFEval": 81.7, "GPQA": 49.0, "AlpacaEval LC": 50.5,
 },
 "Llama-3.1-70B": {
 "MMLU": 86.0, "GSM8K": 91.2, "HumanEval": 80.5, "HellaSwag": 85.0,
 "IFEval": 80.4, "GPQA": 46.7, "AlpacaEval LC": 44.8,
 },
 "Qwen2.5-7B (reference)": {
 "MMLU": 74.3, "GSM8K": 83.4, "HumanEval": 68.9, "HellaSwag": 82.1,
 "IFEval": 68.7, "GPQA": 36.7, "AlpacaEval LC": 38.5,
 },
}

datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA", "AlpacaEval LC"]

# compare
print("modelevaluationcompare (, ) \n")

header = f"{'model':<22s}"
for ds in datasets:
 header += f" {ds:>13s}"
print(header)
print("-" * (22 + 14 * len(datasets)))

for model, scores in benchmark_results.items():
 row = f"{model:<22s}"
 for ds in datasets:
 score = scores.get(ds)
 if score is None:
 row += f" {'N/A':>13s}"
 else:
 row += f" {score:>13.1f}"
 print(row)

#
print("\n\n")
print("1. compare (7B vs 7B) , compare (7B vs 671B) ")
print("2. metric: API IFEval, MMLU")
print("3. dataset, ")
print("4. evaluation (prompt/few-shot/seed) , ")
print("5. model; model, , benchmark evaluation")

### 5.1 Visualizing Evaluation Results

. leaderboard :

| | | |
|:---|:---|:---|
| ** (Radar/Spider) ** | compare 2-4 model | OpenAI/DeepSeek report |
| **** | comparemodeldatasetscore | ablation |
| **** | LLM-as-Judge compareresult | Chatbot Arena |

In [ ]:
# + + : visualize
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# ============================================
# 1: (Spider/Radar Chart)
# ============================================
radar_models = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (reference)"]
radar_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]
radar_colors = ["#2563EB", "#10B981", "#F59E0B", "#EF4444"]

radar_values = []
for model in radar_models:
 vals = [benchmark_results[model][ds] for ds in radar_datasets]
 radar_values.append(vals)

num_vars = len(radar_datasets)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1] #

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model, values, color in zip(radar_models, radar_values, radar_colors):
 values_closed = values + values[:1]
 ax.fill(angles, values_closed, alpha=0.05, color=color)
 ax.plot(angles, values_closed, "o-", linewidth=2, color=color, label=model, markersize=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_datasets, fontsize=11)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=8, color="gray")
ax.set_title("Model Comparison — Radar Chart", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(": , ")
print(", model\n")

# ============================================
# 2: (Grouped Bar Chart)
# ============================================
bar_models = radar_models
bar_datasets = radar_datasets

x = np.arange(len(bar_datasets))
width = 0.2
n_models = len(bar_models)

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model, color) in enumerate(zip(bar_models, radar_colors)):
 values = [benchmark_results[model][ds] for ds in bar_datasets]
 offset = width * (i - n_models/2 + 0.5)
 bars = ax.bar(x + offset, values, width, label=model, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
 for bar, val in zip(bars, values):
 ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f"{val:.1f}",
 ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.set_xlabel("Benchmark", fontsize=12)
ax.set_ylabel("Score (%)", fontsize=12)
ax.set_title("Model Comparison — Grouped Bar Chart", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(bar_datasets, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(": compare, ablation ")
print("model () benchmark model\n")

# ============================================
# 3: (LLM-as-Judge )
# ============================================
print("--- LLM-as-Judge ---\n")

models_for_matrix = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (reference)"]
n_mat = len(models_for_matrix)
# : model vs model
win_rate_matrix = np.array([
 [0.50, 0.55, 0.62, 0.85],
 [0.45, 0.50, 0.57, 0.80],
 [0.38, 0.43, 0.50, 0.72],
 [0.15, 0.20, 0.28, 0.50],
])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(win_rate_matrix, cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(n_mat))
ax.set_yticks(range(n_mat))
ax.set_xticklabels(models_for_matrix, fontsize=10, rotation=30, ha="right")
ax.set_yticklabels(models_for_matrix, fontsize=10)
ax.set_title("Pairwise Win Rate Matrix\n(row model vs col model)", fontsize=12, fontweight="bold")

for i in range(n_mat):
 for j in range(n_mat):
 color = "white" if win_rate_matrix[i][j] < 0.3 or win_rate_matrix[i][j] > 0.7 else "black"
 ax.text(j, i, f"{win_rate_matrix[i][j]:.2f}", ha="center", va="center", fontsize=11, fontweight="bold", color=color)

plt.colorbar(im, ax=ax, label="Win Rate")
plt.tight_layout()
plt.show()

print(": model vs model, >0.5 model")
print("Qwen2.5-7B compare < 0.5, Notemodel")

In [ ]:
# composite score: methodcompare
import numpy as np
from scipy import stats

print("=== composite score ===\n")

# benchmark_results
scores = benchmark_results

# benchmark ( AlpacaEval LC 100-based)
eval_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]

print("### methodcompare\n")

# --- 1. ---
print("1. (Arithmetic Mean)")
print(f"{'model':<22s} {'Avg':>6s} {'Std':>6s}")
print("-" * 36)
for model in scores:
 vals = [scores[model][ds] for ds in eval_datasets]
 avg = np.mean(vals)
 std = np.std(vals)
 print(f"{model:<22s} {avg:>6.1f} {std:>6.1f}")

# --- 2. ---
print(f"\n2. (Geometric Mean) — ")
print(f"{'model':<22s} {'GMean':>6s}")
print("-" * 30)
for model in scores:
 vals = [scores[model][ds] for ds in eval_datasets]
 gmean = stats.gmean(vals)
 print(f"{model:<22s} {gmean:>6.1f}")

# --- 3. ---
print(f"\n3. (Weighted) — +")
weights = {"MMLU": 0.25, "GSM8K": 0.20, "HumanEval": 0.20, "HellaSwag": 0.10, "IFEval": 0.15, "GPQA": 0.10}
print(f" : {weights}")
print(f"{'model':<22s} {'Weighted':>8s}")
print("-" * 32)
for model in scores:
 weighted = sum(scores[model][ds] * weights[ds] for ds in eval_datasets)
 print(f"{model:<22s} {weighted:>8.1f}")

# --- 4. a strong model ---
print(f"\n4. (Normalized) — GPT-4o 100% ")
baseline = "GPT-4o"
print(f"{'model':<22s}", end="")
for ds in eval_datasets:
 print(f" {ds:>8s}", end="")
print(f" {'Avg%':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 8))
for model in scores:
 normalized = [scores[model][ds] / scores[baseline][ds] * 100 for ds in eval_datasets]
 avg_norm = np.mean(normalized)
 print(f"{model:<22s}", end="")
 for nv in normalized:
 print(f" {nv:>8.1f}", end="")
 print(f" {avg_norm:>8.1f}")

# --- 5. ---
print(f"\n5. (Rank Sum) — ")
models_list = list(scores.keys())
ranks = {ds: np.argsort([-scores[m][ds] for m in models_list]).argsort() + 1 for ds in eval_datasets}
print(f"{'model':<22s}", end="")
for ds in eval_datasets:
 print(f" {ds:>8s}", end="")
print(f" {'Sum':>6s} {'AvgRank':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 14))
for i, model in enumerate(models_list):
 rank_list = [ranks[ds][i] for ds in eval_datasets]
 rank_sum = sum(rank_list)
 rank_avg = np.mean(rank_list)
 print(f"{model:<22s}", end="")
 for r in rank_list:
 print(f" {r:>8.0f}", end="")
 print(f" {rank_sum:>6.0f} {rank_avg:>8.1f}")

# --- ---
print(f"\n### composite scorerecommended")
print(" : + () ")
print(" PPT: 2-3 method, ")
print(" : ( = ) ")
print(" : ( HuggingFace Leaderboard) ")
print(" , model")

### 5.2 Computing a Composite Score

datasetscore, "". method:

| method | | |
|:---|:---|:---|
| ** (Avg)** | $\frac{1}{N}\sum s_i$ | compare, |
| **** | $\frac{1}{N}\sum \frac{s_i}{\max(s_i)}$ | benchmark |
| **** | $\sum w_i s_i$ | () |
| **** | $(\prod s_i)^{1/N}$ | , |
| **** | $\sum rank_i$ | , |
| **Elo ** | compare | LLM-as-Judge |

****:
- : model 90 10 , 50 ,
- ****: , ; ,
- : /, 

## 6. AlpacaEval Hands-On

 LLM-as-Judge , prompt result. AlpacaEval pipeline.

### 6.1 AlpacaEval

```
1. 805 prompt
2. modelgeneration (OpenAI-Compatible API)
3. GPT-4 : vs ,
4. output: Win Rate / LC Win Rate / Avg Length
```

### 6.2

```bash
# Step 1: generationmodel
alpaca_eval evaluate_from_model \
 --model_name_or_path "your-model" \
 --output_path results/your-model \
 --max_instances 100 # 100

# Step 2: GPT-4
export OPENAI_API_KEY="sk-xxx"
alpaca_eval evaluate \
 --annotators_config "alpaca_eval_gpt4_turbo_fn" \
 --model_outputs "results/your-model.json" \
 --output_path "results/your-model-eval"

# Step 3: result
cat results/your-model-eval/leaderboard.csv
```

### 6.3 Python API (Notebook run)

```python
from alpaca_eval import evaluate
import pandas as pd

df = evaluate(
 model_outputs="results/your-model-outputs.json",
 annotators_config="alpaca_eval_gpt4_turbo_fn",
 max_instances=100,
)
print(f"Win Rate: {df['win_rate'].iloc[0]:.1%}")
print(f"LC Win Rate: {df['lc_win_rate'].iloc[0]:.1%}")
print(f"Avg Length: {df['avg_length'].iloc[0]:.0f} chars")
```

### 6.4 AlpacaEval result

| metric | | |
|:---|:---|:---|
| **Win Rate** | GPT-4 | leaderboard, baseline, annotator model |
| **LC Win Rate** | bias | , raw WR; |
| **Avg Length** | | , WR |

### 6.5 OpenAI-Compatible API AlpacaEval

modeldeployment OpenAI-Compatible API, :

```python
from openai import OpenAI
import json

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
with open("alpaca_eval/prompts/alpaca_eval.json") as f:
 prompts = json.load(f)

outputs = []
for item in prompts[:100]: # 100
 resp = client.chat.completions.create(
 model="your-model",
 messages=[{"role": "user", "content": item["instruction"]}],
 temperature=0,
 max_tokens=1024,
 )
 outputs.append({
 "instruction": item["instruction"],
 "output": resp.choices[0].message.content,
 "generator": "your-model",
 })

with open("results/your-model-outputs.json", "w") as f:
 json.dump(outputs, f, ensure_ascii=False, indent=2)
print(f"generation {len(outputs)} , : alpaca_eval evaluate ...")
```

## 7. Domain-Specific Evaluation

### 7.1 RAG evaluation

 RAG , benchmark , evaluation:

| | metric | |
|:---|:---|:---|
| **** | Context Precision / Recall | |
| **** | NDCG / MRR | |
| **generation** | Faithfulness () | |
| **generation** | Answer Relevance | |
| **** | Noise Sensitivity | , |

recommended:

```bash
pip install ragas # RAG evaluation
```

: . generation, ——model.

### 7.2 modelevaluation: pass@k

| metric | | |
|:---|:---|:---|
| **pass@1** | generation 1 , | ( 1 output) |
| **pass@k** | generation k , 1 | () |
| **repeated pass / all-pass@k () ** | generation k , | ; pass@k |

HumanEval pass@k, SWE-bench resolved rate. "correct", repeated pass rate all-pass@k, metric. 

## 8. Bias and Consistency of LLM-as-Judge

### 8.1 LLM-as-Judge bias

| bias | | method |
|:---|:---|:---|
| **Position Bias** | | , |
| **Length Bias** | | LC (Length-Controlled) Win Rate |
| **Egocentric Bias** | Judge training, , | judgemodel (GPT-4 + Claude + open-source judge) |
| **Verbosity Bias** | | judge + |

:

- compare
- ; , evaluation,
- judge model

### 8.2 consistencyevaluation

. model 85%, answer——model.

| metric | | |
|:---|:---|:---|
| **CR@K (Consistency Rate)** | K , | answer |
| **Prompt Robustness** | , answer | prompt |
| **Order Robustness** | , answer | bias |
| **Sampling Robustness** | temperature 0.3, output | deployment |

```
 + consistency = , , , ,
 + consistency = ()
```

## 9. evaluationmetric

### 9.1 metric

| | evaluation | metric | dataset |
|:---|:---|:---|:---|
| **** | `loglikelihood`: , | `acc` / `acc_norm` | MMLU, HellaSwag, ARC |
| **generation** | `generate_until`: generation, answer | `exact_match` / `pass@k` / `F1` | GSM8K, HumanEval |
| **** | LLM-as-Judge: GPT-4 | `win_rate` / `Elo` / `score` | MT-Bench, AlpacaEval |
| **** | check: , , | `strict_acc` | IFEval |
| **** | logprob | `ppl` / `bpb` | WikiText, Lambada |

### 9.2 modelreportmetric

| metric | | |
|:---|:---|:---|
| **IFEval strict** | check, , | "" |
| **GPQA** | Google-Proof Q&A, | model |
| **AIME pass@1** | 15 , | GSM8K |
| **SWE-bench** | GitHub issue → bug → | Agent |
| **LiveCodeBench** | LeetCode/Codeforces | |
| **RULER** | ( 128K tokens) | Needle-in-Haystack |

### 9.3 metric

```
model？
├── → MT-Bench + AlpacaEval 2.0
├── → HumanEval+ + LiveCodeBench + SWE-bench
├── → GSM8K + MATH + AIME 2024
├── → MMLU-Pro + GPQA
├── → IFEval + MT-Bench ()
├── RAG → RAGAS ( + + )
└── Agent → SWE-bench + WebArena + ToolBench
```

## 10.

### 10.1

| | | Note | |
|:---|:---|:---|:---|
| **** | | trainingevaluation, score | LiveCodeBench dataset; trainingevaluation exact / n-gram / embedding decontamination check. `--check_integrity` check, . |
| **Prompt ** | | model, prompt score 5-15% | OLMES prompt; prompt version |
| **Few-shot ** | | 0-shot/5-shot/8-shot result | . MMLU 5-shot, GSM8K 8-shot, report 0-shot, CoT, maj@k . |
| **evaluation** | | HumanEval model"" | HumanEval+ LiveCodeBench |
| **dataset** | | HumanEval 164 , | dataset |

### 10.2

| | Note | |
|:---|:---|:---|
| **Chat vs Completions API ** | MMLU Chat API | Completions API, generation Chat API |
| **batch size ** | OOM, | vLLM 8-16, HF auto |
| **seed ** | result | `--model_args seed=42` |
| **temperature 0** | evaluation temperature=0; pass@k, consistency, creative writing robustness evaluation temperature | `temperature=0` |
| **** | API evaluation | `num_concurrent=4-8` ( API ) |

### 10.3

| | Note |
|:---|:---|
| **score** | reportrun ± |
| **compare** | evaluation (prompt/few-shot/) , |
| **** | MMLU 57 , |
| **bias** | model, LLM-as-Judge —— LC Win Rate |

## 11. hands-on

```bash
# === (30 ) ===
#
pip install lm-eval openai

# DeepSeek API GSM8K (generation, Chat API)
lm_eval --model local-chat-completions \
 --model_args model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4,tokenized_requests=False \
 --tasks gsm8k --batch_size 8 --limit 50 \
 --output_path ./eval_results/

# vLLM deploymentopen-sourcemodel MMLU (, Completions API)
lm_eval --model local-completions \
 --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/completions,num_concurrent=4,tokenized_requests=False \
 --tasks mmlu --batch_size 16 --limit 100 \
 --output_path ./eval_results/

# HuggingFace model
lm_eval --model hf \
 --model_args pretrained=Qwen/Qwen2.5-7B-Instruct,dtype=bfloat16 \
 --tasks gsm8k,mmlu,hellaswag --batch_size auto \
 --output_path ./eval_results/

# === dataset ===
lm_eval --tasks list | head -30

# === Python API ( Notebook/) ===
# from lm_eval import simple_evaluate
# results = simple_evaluate(
# model='local-chat-completions',
# model_args='model=deepseek-chat,base_url=...,token=...',
# tasks=['gsm8k', 'ifeval'],
# limit=50,
# )
```

## 12. 2 :

evaluation: model A 82%, model B 80%——A B ？. 2 , evaluation: 100 , , A B .

evaluation: ****. : ？model？, ？？ numpy, GPU. 

### 12.1 : ,

 = / . model""****, —— 100 , model 83 , 79 .

method **bootstrap**: evaluationresult, , . (confidence interval) . 

In [ ]:
import numpy as np
np.random.seed(42)

# model 500 (1=, 0=)
N = 500
results = (np.random.rand(N) < 0.80).astype(int)
acc = results.mean()

# bootstrap: 2000 ,
B = 2000
boot_acc = np.array([np.random.choice(results, N, replace=True).mean()
 for _ in range(B)])
ci = np.percentile(boot_acc, [2.5, 97.5])

print(f" = {acc:.3f}")
print(f"95% = [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f" = ±{(ci[1]-ci[0])/2*100:.1f}%")
print("\n: 0.79, . ")
print(" reportmodel CI, . ")

### 12.2 : McNemar

model, wrong, . model****, ——. correct**result**: A B , A B . .

**McNemar **. "": $b$ = AB, $c$ = AB. $\chi^2 = (|b-c|-1)^2 / (b+c)$, 3.84 ($\alpha=0.05$) . 

In [ ]:
np.random.seed(7)
N = 500
# model (, result)
difficulty = np.random.randn(N)
a = (difficulty + np.random.randn(N) * 0.6 < 0.85).astype(int) # model A
b = (difficulty + np.random.randn(N) * 0.6 < 0.78).astype(int) # model B

acc_a, acc_b = a.mean(), b.mean()
n12 = int(((a == 1) & (b == 0)).sum()) # A B
n21 = int(((a == 0) & (b == 1)).sum()) # A B
chi2 = (abs(n12 - n21) - 1) ** 2 / (n12 + n21)
significant = chi2 > 3.84

print(f"model A = {acc_a:.3f} model B = {acc_b:.3f}")
gap = (acc_a - acc_b) * 100
who = "" if gap >= 0 else ""
print(f"A B {who} {abs(gap):.1f} ")
print(f"AB = {n12} AB = {n21}")
print(f"χ² = {chi2:.2f} 3.84？ {significant}")
print(f"\n: A B {who} {abs(gap):.1f} , χ²={chi2:.2f} 3.84 → . ")
print(" model. , . ")

### 12.3 consistency: Cohen's κ

LLM-as-Judge ( 9 ) evaluation, "/""/". , ？metric**** (agree ) , —— 90% sample"correct", 80%+ .

**Cohen's kappa (κ) ** : $\kappa = (p_o - p_e)/(1 - p_e)$, $p_o$ , $p_e$ "". κ=1 , κ=0 . 

In [ ]:
np.random.seed(11)
N = 200
true_label = np.random.randint(0, 2, N)
# ,
ann1 = np.where(np.random.rand(N) < 0.10, 1 - true_label, true_label)
ann2 = np.where(np.random.rand(N) < 0.15, 1 - true_label, true_label)

p_o = (ann1 == ann2).mean() #
p_e = sum((ann1 == k).mean() * (ann2 == k).mean() for k in [0, 1])
kappa = (p_o - p_e) / (1 - p_e)

print(f" p_o = {p_o:.3f}")
print(f" p_e = {p_e:.3f}")
print(f"Cohen's κ = {kappa:.3f}")
print("\n (Landis & Koch) : ")
print(" κ<0.20 | 0.40-0.60 | 0.60-0.80 | >0.80 ")
print(f"\n: {p_o:.2f} , κ {kappa:.2f} () . ")
print(" report LLM-as-Judge , κ, . ")

### 12.4 : 2%

: model 2%, ？**** (power) .

: , . sample $n \approx (z_\alpha + z_\beta)^2 \cdot 2p(1-p) / d^2$, $d$ . """". 

In [ ]:
import matplotlib.pyplot as plt

z_a, z_b = 1.96, 0.84 # α=0.05, power=0.8
p = 0.5 #
ds = np.linspace(0.01, 0.12, 50)
ns = (z_a + z_b) ** 2 * 2 * p * (1 - p) / ds ** 2

plt.figure(figsize=(7, 3.5))
plt.plot(ds * 100, ns, color="steelblue")
plt.yscale("log")
plt.xlabel("Detectable accuracy gap (%)")
plt.ylabel("Examples needed per model (log)")
plt.title("Sample size vs detectable gap (power=0.8)")
plt.grid(True, alpha=0.3)
plt.show()

for d in [0.10, 0.05, 0.02]:
 n = (z_a + z_b) ** 2 * 2 * p * (1 - p) / d ** 2
 print(f" {d*100:.0f}% → {int(n):,} /model")

print("\n: 100 benchmark 10% ; ")
print(" 2% , . evaluation, . ")

### 12.5 evaluation,

check. modelcompare, :

- ** / ？** , score
- **model？** (McNemar) ,
- **evaluation？** 100 2% , (12.4)
- **evaluation？** LLM-as-Judge , κ, (12.3)

trainingmodel, : **score, . ** CMU ""——, score. 

## Summary

### ()

| # | | |
|:---|:---|:---|
| 1 | evaluation | 2025 /, evaluation, evaluation |
| 2 | Repo | lm-eval-harness, AlpacaEval, FastChat, DeepEval |
| 3 | OpenAI-Compatible API | `local-chat-completions` vs `local-completions`, API |
| 4 | LLM-as-Judge | MT-Bench prompt + OpenAI SDK |
| 5 | resultaggregatevisualize | compare + + + + 5 composite scoremethod |
| 6 | AlpacaEval hands-on | CLI + Python API , OpenAI-Compatible |
| 7 | evaluation | RAG (RAGAS /) , (pass@k ) |
| 8 | LLM-as-Judge biasconsistency | Position/Length/Egocentric Bias + CR@K/Prompt Robustness |
| 9 | metric | acc / exact_match / pass@k / win_rate / Elo, 2025 metric (AIME, SWE-bench, LiveCodeBench) |
| 10 | | (/prompt /few-shot) , (API /seed/temperature) , |
| 11 | | CLI + Python API, run |

### recommended Repo aggregate

```bash
# evaluationframework
git clone https://github.com/EleutherAI/lm-evaluation-harness.git # open-sourceevaluationframework,
git clone https://github.com/tatsu-lab/alpaca_eval.git # LLM-as-Judge, 805 prompt
git clone https://github.com/lm-sys/FastChat.git # MT-Bench + Chatbot Arena

#
pip install deepeval # CI/CD evaluation (, G-Eval, 40+ metric)
pip install ragas # RAG evaluation (, )
```

###

```
Level 1 () :
 1. pip install lm-eval openai
 2. DeepSeek API gsm8k --limit 50
 3. output JSON

Level 2 () :
 1. vLLM deploymentopen-sourcemodel
 2. 4 dataset: gsm8k + mmlu + humaneval + ifeval
 3. +
 4. leaderboard score

Level 3 () :
 1. AlpacaEval / MT-Bench evaluation
 2. evaluation (RAGAS / prompt)
 3. consistencymetric (CR@K) ,
```

###

| | |
|:---|:---|
| **loglikelihood vs generate_until** | (Completions API) , generation (Chat API) |
| **pass@k ** | pass@k (k 1 ) ; repeated pass / all-pass@k |
| **LC Win Rate** | , "" |
| ** vs ** | , model |
| **CR@K** | K , —— |
| **Position / Length / Egocentric Bias** | LLM-as-Judge bias, + LC + |
| **** | trainingevaluationscore—— LiveCodeBench dataset |

: evaluationmodeldeployment. evaluation = + + score + . 

##

1. **Perplexity **

 3-token , modeloutput logit (softmax ) [0.5, 0.3, 0.2], [0.1, 0.7, 0.2], [0.4, 0.1, 0.5], perplexity.

 <details><summary></summary> token log , , exp. Perplexity Notemodel"". </details>

2. **LLM-as-Judge bias**

 : (answer_a answer_b ) , 3 , GPT-4 . Position Bias .

 <details><summary></summary>Position Bias = / . Notejudge. </details>

3. **evaluationreport**

 , model 4 benchmark , .

 ```python
 scores = {
 'Model A': {'gsm8k': 82, 'mmlu': 71, 'humaneval': 65, 'ifeval': 78},
 'Model B': {'gsm8k': 78, 'mmlu': 75, 'humaneval': 70, 'ifeval': 72},
 }
 ```

 <details><summary></summary> = (a * b * c * d) ** (1/4). comparemodel, model"". </details>

In [ ]:
# 1: Perplexity ()
import math

# token softmax
probs = [[0.5, 0.3, 0.2], [0.1, 0.7, 0.2], [0.4, 0.1, 0.5]]
# "modelcorrect token "
# : per_token_prob = ; avg_log_prob = mean(log(per_token_prob))
per_token_prob = None # [0.5, 0.7, 0.5]
avg_log_prob = None # log
perplexity = None # exp(-avg_log_prob)

assert per_token_prob is not None, " per_token_prob"
assert avg_log_prob is not None, " avg_log_prob"
assert perplexity is not None, " perplexity"

expected_probs = [max(p) for p in probs] # [0.5, 0.7, 0.5]
assert per_token_prob == expected_probs, "per_token_prob "
expected_avg = sum(math.log(p) for p in expected_probs) / len(expected_probs)
assert abs(avg_log_prob - expected_avg) < 1e-6, "avg_log_prob "
expected_ppl = math.exp(-expected_avg)
assert abs(perplexity - expected_ppl) < 1e-4, "perplexity "

print(f" token : {per_token_prob}")
print(f" log : {avg_log_prob:.4f}")
print(f"Perplexity: {perplexity:.3f}")
print("✅ 1 : Perplexity = exp(- log ), Notemodel. ")

In [ ]:
# 2: Position Bias ()
# 3 : answer_a vs answer_b .
# , Notejudge.
# : Position Bias = /

# scores_before / scores_after [answer_a , answer_b ],
experiments = [
 ([8.0, 6.0], [7.5, 6.5]), # a →
 ([7.0, 9.0], [9.5, 7.0]), # b , a →
 ([6.0, 5.0], [5.5, 5.0]), # a →
]

def winner(scores):
 # 0 answer_a , 1 answer_b
 return 0 if scores[0] >= scores[1] else 1

# TODO: "" ()
inconsistent = None

assert inconsistent is not None, ""
total = len(experiments)
expected = sum(1 for before, after in experiments if winner(before) != winner(after))
assert inconsistent == expected, f" {expected}"

position_bias_rate = inconsistent / total
assert 0 <= position_bias_rate <= 1, "Position Bias 0 1 "

print(f": {total}, : {inconsistent}")
print(f"Position Bias : {position_bias_rate:.0%}")
if position_bias_rate > 0:
 print("Note: judge. ")
else:
 print("Note: bias. ")
print("✅ 2 : Position Bias , judge. ")

In [ ]:
# 3: ()
import math

scores = {
 'Model A': {'gsm8k': 82, 'mmlu': 71, 'humaneval': 65, 'ifeval': 78},
 'Model B': {'gsm8k': 78, 'mmlu': 75, 'humaneval': 70, 'ifeval': 72},
}

# TODO: model
# : = (a * b * c * d) ** (1/4), math.prod
geo_mean = {} # {'Model A': ..., 'Model B': ...}

assert geo_mean is not None, " geo_mean"
assert len(geo_mean) == 2, "model"

for name, m in scores.items():
 vals = list(m.values())
 expected = (math.prod(vals)) ** (1 / len(vals))
 assert abs(geo_mean[name] - expected) < 1e-6, f"{name} "

print(f": {geo_mean}")
print(f"Model A : {geo_mean['Model A'] > geo_mean['Model B']}")
print(": Model B , Model A humaneval . ")
print("✅ 3 : modelevaluation. ")

## 13. hands-on: lm-eval evaluationtrainingmodel

 2 , 9.34M model loss , output. "output"
"": , ; , Note.
, ？

training, :

1. **framework**: lm-eval Hugging Face model HellaSwag
2. **evaluation**: 64M model GSM8K result
3. ****: n-gram
4. ****: evaluation, model

> 🔬 **From-0 hands-on 3/3**: 1 training Tokenizer, 2 PT/SFT,
> model. 

### 13.1 evaluationframework

**lm-eval (EleutherAI lm-evaluation-harness) **: APImodel, benchmark,
metricopen-sourceevaluationframework. , : GSM8K,
MMLU, HellaSwag run, open-sourcecommunityreportresult.

 `hf-internal-testing/tiny-random-LlamaForCausalLM` HellaSwag. model,
, "model → → metric".

**HellaSwag**: , model. 0-shot,
 42, 1%. `limit` pipeline, . 

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from lm_eval import simple_evaluate

smoke_result = simple_evaluate(
 model="hf",
 model_args={
 "pretrained": "hf-internal-testing/tiny-random-LlamaForCausalLM",
 "dtype": "float32",
 },
 tasks=["hellaswag"],
 num_fewshot=0,
 batch_size=4,
 device="cuda:0",
 limit=0.01,
 bootstrap_iters=0,
 log_samples=False,
 verbosity="ERROR",
 random_seed=42,
 numpy_random_seed=42,
 torch_random_seed=42,
 fewshot_random_seed=42,
)

hellaswag = smoke_result["results"]["hellaswag"]
effective_samples = smoke_result["n-samples"]["hellaswag"]["effective"]
smoke_scores = [hellaswag["acc,none"], hellaswag["acc_norm,none"]]

print(f"runsample: {effective_samples}")
print(f"acc: {smoke_scores[0]:.2%}")
print(f"acc_norm: {smoke_scores[1]:.2%}")
print(": model; evaluation, model. ")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.bar(["Accuracy", "Length-normalized accuracy"], smoke_scores,
 color=["#4C78A8", "#F58518"])
ax.axhline(0.25, color="#777777", linestyle="--", label="Random-choice baseline")
ax.bar_label(bars, fmt="%.2f")
ax.set_ylim(0, max(smoke_scores + [0.25]) * 1.35)
ax.set_ylabel("Score")
ax.set_title("HellaSwag Smoke Test (1% Validation Split)")
ax.legend()
plt.show()

`acc` compare; `acc_norm` answer log-likelihood,
answer. 25%, sample, model.

:

```bash
lm_eval --model hf \
 --model_args pretrained=hf-internal-testing/tiny-random-LlamaForCausalLM,dtype=float32 \
 --tasks hellaswag --num_fewshot 0 --batch_size 4 --limit 0.01 --seed 42
```

 lm-eval score, ？result:

| | | |
|:---|:---|:---|
| version | `hellaswag`, lm-eval 0.4.12 | prompt, answer |
| metric | `acc` / `acc_norm` | |
| few-shot | 0-shot | |

, version, metric, few-shot, prompt sample, result. 

### 13.2 evaluation 64M model: GSM8K

**GSM8K**: 1,319 , modelgenerationinferenceanswer.
, "", checkmodel, answer.

**exact_match,strict-match**: answerresult, answer
. , , answercorrect.

 2 9.34M model Notebook training, SFT checkpoint;
evaluation. 64M model lm-eval result. evaluation 1,319 ,
 `limit` . model, run lm-eval `gsm8k` ;
run JSON, , . 

In [ ]:
report_dir = Path("llm_train/reports")
result_files = sorted(report_dir.glob("eval_gsm8k_*.json"))
assert len(result_files) == 6, " GSM8K result"

short_names = ["A1", "A2", "A3", "B1", "C1", "D1"]
strict_scores = []
sample_counts = []
correct_counts = []

for result_path in result_files:
 with result_path.open(encoding="utf-8") as file:
 task_result = json.load(file)["tasks"]["gsm8k"]
 score = task_result["exact_match,strict-match"]
 count = task_result["sample_len"]
 strict_scores.append(score)
 sample_counts.append(count)
 correct_counts.append(round(score * count))

assert sample_counts == [1319] * 6, ""
assert max(strict_scores) == min(strict_scores), "score"

# Wilson p ± 1.96*SE 0 .
z = 1.96
intervals = []
for correct, count in zip(correct_counts, sample_counts):
 proportion = correct / count
 denominator = 1 + z ** 2 / count
 center = (proportion + z ** 2 / (2 * count)) / denominator
 margin = z * (proportion * (1 - proportion) / count
 + z ** 2 / (4 * count ** 2)) ** 0.5 / denominator
 intervals.append((center - margin, center + margin))

print(f": {len(result_files)}")
print(f": {sample_counts[0]}")
print(f": {correct_counts[0]}")
print(f"strict exact match: {strict_scores[0]:.2%}")
print(f"95% Wilson CI: [{intervals[0][0]:.2%}, {intervals[0][1]:.2%}]")
print(": SFT score, . ")

score_percent = [score * 100 for score in strict_scores]
lower_error = [(score - low) * 100 for score, (low, high) in zip(strict_scores, intervals)]
upper_error = [(high - score) * 100 for score, (low, high) in zip(strict_scores, intervals)]

fig, ax = plt.subplots(figsize=(7.2, 3.6))
bars = ax.bar(short_names, score_percent, color="#4C78A8")
ax.errorbar(short_names, score_percent, yerr=[lower_error, upper_error], fmt="none",
 ecolor="#333333", capsize=4, label="95% Wilson CI")
ax.bar_label(bars, labels=[f"{score:.2f}%" for score in score_percent], padding=3)
ax.set_ylim(0, max(upper_error) + score_percent[0] + 0.25)
ax.set_xlabel("Training recipe")
ax.set_ylabel("Strict exact match (%)")
ax.set_title("GSM8K Results for Six 64M Training Recipes")
ax.legend()
plt.show()

 JSON result. 0.61% 1,319 8 :

| | / SFT | | | exact_match,strict-match |
|:---|:---|---:|---:|---:|
| A1 | scratchpad v2 | 1,319 | 8 | 0.61% |
| A2 | Belle + scratchpad v2 | 1,319 | 8 | 0.61% |
| A3 | Belle + CoT | 1,319 | 8 | 0.61% |
| B1 | Belle | 1,319 | 8 | 0.61% |
| C1 | Belle, block size 1024 | 1,319 | 8 | 0.61% |
| D1 | 10B-token PT + Belle | 1,319 | 8 | 0.61% |

"evaluation", evaluation: , CoT ,
training token , result. , trainingpipeline,
64M model; SFT .
[Scaling Laws ](../part2-training/12-scaling-laws.ipynb): ,
, .

: This chapter 12.4 Note, , .
8/1,319 95% Wilson 0.31%～1.19%. , ;
This chapter 12.2 McNemar . ,
"", modelinput. 

 checkpoint, run. JSON
; modelevaluation, Notebook training.

```bash
HIP_VISIBLE_DEVICES=0 CUDA_VISIBLE_DEVICES=0 \
python llm_train/scripts/run_eval.py \
 --ckpt llm_train/checkpoints/gsm8k_a1_scratchpad_v2/sft.pt \
 --model_type dense --tasks gsm8k \
 --out llm_train/reports/eval_gsm8k_a1_scratchpad_v2.json
```

result: checkpoint, , lm-eval version, few-shot , prompt/chat template,
samplemetric. "GSM8K 0.61%". 

### 13.3 score:

GSM8K exact match answercorrect, model. model,
output, . generationmetric.

** n-gram **: $n$ ,
. , , .

****: . ,
model. , , ,
trigram ($n=3$) . metric, . 

In [ ]:
def repetition_diagnostics(text, n=3):
 """ n-gram .

 :
 text: generation.
 n: n-gram .
 :
 , .
 """
 units = re.findall(r"[\u4e00-\u9fff]|[A-Za-z]+|\d+|[^\s]", text)
 ngrams = [tuple(units[i:i + n]) for i in range(len(units) - n + 1)]
 counts = Counter(ngrams)
 repeated = sum(count - 1 for count in counts.values() if count > 1)
 ratio = repeated / len(ngrams) if ngrams else 0.0

 longest = ()
 for length in range(1, len(units) // 2 + 1):
 first_position = {}
 found = ()
 for start in range(len(units) - length + 1):
 fragment = tuple(units[start:start + length])
 if fragment in first_position and start - first_position[fragment] >= length:
 found = fragment
 break
 first_position.setdefault(fragment, start)
 if not found:
 break
 longest = found

 return {
 "repeat_ngram_ratio": ratio,
 "longest_fragment": "".join(longest),
 "longest_fragment_units": len(longest),
 }

In [ ]:
generation_samples = {
 "9.34M / 150-step SFT": (
 ", 》, "
 ),
 "64M repetition failure": (
 ", , . "
 ", . "
 ),
}

diagnostics = {
 name: repetition_diagnostics(text, n=3)
 for name, text in generation_samples.items()
}

for name, result in diagnostics.items():
 print(name)
 print(f" trigram : {result['repeat_ngram_ratio']:.2%}")
 print(f" : {result['longest_fragment']!r}")
 print(f" : {result['longest_fragment_units']} ")

names = list(diagnostics)
ratios = [diagnostics[name]["repeat_ngram_ratio"] * 100 for name in names]
lengths = [diagnostics[name]["longest_fragment_units"] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
ratio_bars = axes[0].bar(names, ratios, color=["#4C78A8", "#E45756"])
axes[0].bar_label(ratio_bars, fmt="%.1f%%")
axes[0].set_ylabel("Repeated trigram ratio (%)")
axes[0].set_title("Local Repetition Rate")
axes[0].tick_params(axis="x", rotation=12)

length_bars = axes[1].bar(names, lengths, color=["#4C78A8", "#E45756"])
axes[1].bar_label(length_bars)
axes[1].set_ylabel("Units")
axes[1].set_title("Longest Repeated Fragment")
axes[1].tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.show()

print(": 64M ; modelcorrect. ")

 exact match : 64M generation"", trigram
; 9.34M , .

"""". evaluation:

1. benchmark score""
2. ""
3. generation"model"

### 13.4 hands-on

1. **modelresult. ** 64M model,
 . 0.61% training, model,
 , SFT .
2. **evaluation. ** , few-shot, prompt, answer, chat
 template samplescore. metric.
3. **. ** exact match , ""
 "generation". , traininggoal.

### hands-onSummary (checklist)

:

1. ✅ lm-eval APImetric, modelresult
2. ✅ `limit=0.01` ,
3. ✅ resultversion, metric, few-shot, prompt sample
4. ✅ 64M model GSM8K 8/1,319, strict exact match 0.61%
5. ✅ , "", ""
6. ✅ n-gram , correctevaluation
7. ✅ evaluationreport, , run

> 🔬 **From-0 hands-on 3/3 **
>
> [02 BPE Tokenizer](../part1-foundation/02-bpe-tokenizer.ipynb)
> → [11 PT / SFT](../part2-training/11-training-loss.ipynb)
> → **28 evaluation**
>
> "training Tokenizer → pre-training → SFT → evaluation". 